# 🧭 Akbarxon AI Living Advisor — Uzbekistan

**Author:** Akbarxon Nasirov  
**Mentor:** Dr. Qingyang Xiao

This Google Colab notebook accompanies the GitHub/Streamlit project for an **AI-based living advisor platform**. Users describe their ideal lifestyle in natural language, and the system recommends cities in Uzbekistan, explains the ranking, visualizes results on a map, and learns from like/dislike feedback.

The current repository revision includes a city-specific photo for each of the 16 city profiles, fixes visible raw HTML in city cards, and preserves the Streamlit Cloud toolbar-overlap repair.

> **Important:** Keep the repository's `assets/`, `.streamlit/`, and `ui_source/` folders beside this notebook when executing the packaging cells. The included city scores are illustrative, not official statistics. A public production app must replace them with licensed, dated, auditable data and show sources and update timestamps.


## 1. Install the required packages

Run this cell first in Google Colab. The notebook is designed for Python 3.12-compatible Colab runtimes.

In [ ]:
%pip install -q \
  "streamlit>=1.58,<2.0" \
  "pandas>=2.2,<3.0" \
  "numpy>=2.0,<3.0" \
  "scikit-learn>=1.6,<2.0" \
  "plotly>=6.0,<7.0" \
  "folium>=0.19,<1.0" \
  "streamlit-folium>=0.24,<1.0" \
  "requests>=2.32,<3.0" \
  "joblib>=1.4,<2.0"

In [ ]:
import platform
import sys
from pathlib import Path

print('Python:', sys.version)
print('Platform:', platform.platform())
print('Working directory:', Path.cwd())

## 2. Create the Uzbekistan city prototype dataset

The values below are intentionally labeled as **prototype scores from 0 to 100**. Higher is better for every dimension, including affordability. They are suitable for software demonstration, not relocation decisions.

In [ ]:
from pathlib import Path
import pandas as pd

rows = [
    {
        "city": "Tashkent", "region": "Tashkent City", "lat": 41.2995, "lon": 69.2401,
        "affordability": 52, "grocery_affordability": 56, "entertainment": 94, "sunset_scenery": 70,
        "safety": 76, "jobs": 96, "internet": 93, "healthcare": 95, "education": 96,
        "heritage": 78, "nature": 66, "climate_comfort": 64, "quietness": 38, "mobility": 96,
        "tags": "capital metro jobs universities hospitals restaurants nightlife shopping remote work international airport museums parks",
        "description": "Uzbekistan's largest urban center, with the strongest mix of jobs, universities, healthcare, public transport, dining, and entertainment, but a higher prototype cost profile and a busier pace."
    },
    {
        "city": "Samarkand", "region": "Samarkand Region", "lat": 39.6542, "lon": 66.9597,
        "affordability": 70, "grocery_affordability": 72, "entertainment": 82, "sunset_scenery": 91,
        "safety": 80, "jobs": 72, "internet": 76, "healthcare": 77, "education": 80,
        "heritage": 99, "nature": 74, "climate_comfort": 70, "quietness": 61, "mobility": 83,
        "tags": "silk road registan history architecture tourism sunset restaurants culture train airport walkable old city",
        "description": "A major Silk Road city combining world-famous architecture, strong tourism activity, scenic evenings, restaurants, universities, and good intercity connections."
    },
    {
        "city": "Bukhara", "region": "Bukhara Region", "lat": 39.7681, "lon": 64.4556,
        "affordability": 74, "grocery_affordability": 76, "entertainment": 72, "sunset_scenery": 92,
        "safety": 82, "jobs": 65, "internet": 72, "healthcare": 72, "education": 73,
        "heritage": 98, "nature": 58, "climate_comfort": 61, "quietness": 73, "mobility": 76,
        "tags": "historic center silk road old city sunset courtyards culture tourism calm affordable markets architecture",
        "description": "A compact historic city with a calm atmosphere, strong cultural identity, memorable sunsets over old architecture, and a relatively affordable prototype profile."
    },
    {
        "city": "Khiva", "region": "Khorezm Region", "lat": 41.3783, "lon": 60.3639,
        "affordability": 77, "grocery_affordability": 78, "entertainment": 63, "sunset_scenery": 97,
        "safety": 84, "jobs": 53, "internet": 66, "healthcare": 62, "education": 61,
        "heritage": 100, "nature": 56, "climate_comfort": 57, "quietness": 82, "mobility": 62,
        "tags": "itchan kala walled city heritage sunset photography quiet tourism traditional architecture walkable",
        "description": "A small walled heritage city especially strong for historic atmosphere, photography, sunset views, walkability, and a slower lifestyle."
    },
    {
        "city": "Nukus", "region": "Karakalpakstan", "lat": 42.4600, "lon": 59.6166,
        "affordability": 84, "grocery_affordability": 82, "entertainment": 52, "sunset_scenery": 80,
        "safety": 79, "jobs": 56, "internet": 65, "healthcare": 64, "education": 66,
        "heritage": 76, "nature": 68, "climate_comfort": 48, "quietness": 84, "mobility": 58,
        "tags": "savitsky museum karakalpak culture desert quiet affordable art remote regional center",
        "description": "A quieter and more affordable regional capital known for distinctive art and Karakalpak culture, with a remote desert setting and fewer big-city services."
    },
    {
        "city": "Fergana", "region": "Fergana Region", "lat": 40.3894, "lon": 71.7870,
        "affordability": 78, "grocery_affordability": 83, "entertainment": 69, "sunset_scenery": 78,
        "safety": 82, "jobs": 70, "internet": 74, "healthcare": 75, "education": 76,
        "heritage": 69, "nature": 86, "climate_comfort": 74, "quietness": 70, "mobility": 73,
        "tags": "fergana valley green parks markets family friendly affordable food regional services nature",
        "description": "A greener regional city in the Fergana Valley with strong markets, affordable groceries, family-oriented neighborhoods, and good access to valley destinations."
    },
    {
        "city": "Andijan", "region": "Andijan Region", "lat": 40.7821, "lon": 72.3442,
        "affordability": 80, "grocery_affordability": 85, "entertainment": 66, "sunset_scenery": 72,
        "safety": 80, "jobs": 72, "internet": 73, "healthcare": 74, "education": 75,
        "heritage": 71, "nature": 77, "climate_comfort": 71, "quietness": 66, "mobility": 72,
        "tags": "fergana valley commerce markets affordable groceries family business regional center parks",
        "description": "A commercially active valley city with strong local markets, a relatively affordable prototype cost profile, family services, and regional business opportunities."
    },
    {
        "city": "Namangan", "region": "Namangan Region", "lat": 40.9983, "lon": 71.6726,
        "affordability": 81, "grocery_affordability": 84, "entertainment": 67, "sunset_scenery": 76,
        "safety": 82, "jobs": 69, "internet": 72, "healthcare": 73, "education": 74,
        "heritage": 73, "nature": 83, "climate_comfort": 73, "quietness": 71, "mobility": 69,
        "tags": "gardens flowers valley nature affordable markets family calm regional city parks",
        "description": "A large but comparatively calm valley city associated with gardens, markets, family life, and access to greener landscapes."
    },
    {
        "city": "Qarshi", "region": "Qashqadaryo Region", "lat": 38.8606, "lon": 65.7891,
        "affordability": 83, "grocery_affordability": 82, "entertainment": 58, "sunset_scenery": 75,
        "safety": 80, "jobs": 67, "internet": 68, "healthcare": 69, "education": 69,
        "heritage": 68, "nature": 61, "climate_comfort": 55, "quietness": 78, "mobility": 68,
        "tags": "affordable quiet regional center local markets industry railway warm climate",
        "description": "An affordable regional center with a practical, quieter lifestyle, local industry, railway connections, and fewer entertainment options than the major tourism cities."
    },
    {
        "city": "Termez", "region": "Surxondaryo Region", "lat": 37.2242, "lon": 67.2783,
        "affordability": 82, "grocery_affordability": 80, "entertainment": 55, "sunset_scenery": 88,
        "safety": 78, "jobs": 61, "internet": 66, "healthcare": 67, "education": 67,
        "heritage": 84, "nature": 75, "climate_comfort": 46, "quietness": 79, "mobility": 61,
        "tags": "southern city archaeology buddhist heritage amu darya sunset warm climate quiet border region",
        "description": "A southern city with important archaeological heritage, dramatic river and desert light, warm weather, and a slower regional lifestyle."
    },
    {
        "city": "Jizzakh", "region": "Jizzakh Region", "lat": 40.1158, "lon": 67.8422,
        "affordability": 85, "grocery_affordability": 84, "entertainment": 54, "sunset_scenery": 79,
        "safety": 82, "jobs": 60, "internet": 67, "healthcare": 66, "education": 67,
        "heritage": 59, "nature": 88, "climate_comfort": 69, "quietness": 84, "mobility": 72,
        "tags": "mountains zaamin nature hiking affordable quiet highway family fresh air",
        "description": "A practical and affordable city with strong access to mountain and nature destinations, a quieter pace, and convenient east-west road and rail positioning."
    },
    {
        "city": "Gulistan", "region": "Sirdaryo Region", "lat": 40.4897, "lon": 68.7842,
        "affordability": 88, "grocery_affordability": 87, "entertainment": 47, "sunset_scenery": 68,
        "safety": 83, "jobs": 57, "internet": 66, "healthcare": 64, "education": 64,
        "heritage": 48, "nature": 59, "climate_comfort": 62, "quietness": 88, "mobility": 70,
        "tags": "very affordable quiet small city railway agriculture low cost calm",
        "description": "A smaller, quiet regional capital with one of the strongest illustrative affordability profiles, straightforward transport links, and limited nightlife."
    },
    {
        "city": "Navoi", "region": "Navoi Region", "lat": 40.0844, "lon": 65.3792,
        "affordability": 72, "grocery_affordability": 74, "entertainment": 61, "sunset_scenery": 79,
        "safety": 82, "jobs": 81, "internet": 75, "healthcare": 74, "education": 72,
        "heritage": 57, "nature": 60, "climate_comfort": 58, "quietness": 73, "mobility": 73,
        "tags": "industry mining jobs planned city airport railway parks practical career",
        "description": "A planned industrial city with comparatively strong employment potential, orderly urban form, parks, and useful air and rail links."
    },
    {
        "city": "Urgench", "region": "Khorezm Region", "lat": 41.5500, "lon": 60.6333,
        "affordability": 79, "grocery_affordability": 80, "entertainment": 62, "sunset_scenery": 79,
        "safety": 81, "jobs": 64, "internet": 70, "healthcare": 70, "education": 70,
        "heritage": 76, "nature": 57, "climate_comfort": 56, "quietness": 76, "mobility": 76,
        "tags": "gateway to khiva airport railway affordable markets regional services khorezm",
        "description": "A practical service and transport base for Khorezm, offering airport and rail access, local markets, and convenient proximity to Khiva."
    },
    {
        "city": "Kokand", "region": "Fergana Region", "lat": 40.5286, "lon": 70.9425,
        "affordability": 82, "grocery_affordability": 84, "entertainment": 64, "sunset_scenery": 75,
        "safety": 81, "jobs": 66, "internet": 70, "healthcare": 69, "education": 71,
        "heritage": 88, "nature": 72, "climate_comfort": 71, "quietness": 72, "mobility": 74,
        "tags": "khanate palace heritage valley markets affordable traditional crafts railway",
        "description": "A historic Fergana Valley city with palace architecture, traditional crafts, markets, and a balanced blend of affordability and cultural interest."
    },
    {
        "city": "Shahrisabz", "region": "Qashqadaryo Region", "lat": 39.0578, "lon": 66.8342,
        "affordability": 84, "grocery_affordability": 83, "entertainment": 57, "sunset_scenery": 89,
        "safety": 83, "jobs": 56, "internet": 65, "healthcare": 63, "education": 64,
        "heritage": 93, "nature": 90, "climate_comfort": 72, "quietness": 86, "mobility": 63,
        "tags": "amir timur heritage mountains scenic sunset quiet affordable gardens history",
        "description": "A smaller historic city south of Samarkand with mountain scenery, major Timurid heritage, attractive sunsets, and a quiet lifestyle."
    },
]

output = Path("cities_uzbekistan.csv")
pd.DataFrame(rows).to_csv(output, index=False)
print(f"Wrote {output} with {len(rows)} cities")


In [ ]:
import pandas as pd

cities = pd.read_csv('cities_uzbekistan.csv')
print(f'Loaded {len(cities)} city profiles and {len(cities.columns)} columns.')
display(cities.head(8))

## 3. Generate the complete Streamlit application

The app contains six navigation workspaces:

- **Home:** image-led landing page and featured recommendations
- **Recommend:** natural-language ranking, explanations, city-specific photos, and feedback
- **Map & Compare:** OpenStreetMap/Folium visualization and radar charts
- **Cities:** city-specific photo cards and detailed score profiles for all 16 cities
- **AI Lab:** machine-learning clusters and model explanation
- **About:** project, responsible-use, and deployment information

The card renderer uses compact escaped HTML through `st.html()` so closing tags are never exposed as visible text. No paid API key is required. Optional live-data calls fail safely when unavailable.


In [ ]:
%%writefile app.py
from __future__ import annotations

import base64
import html
import json
import math
import re
from pathlib import Path
from typing import Dict, Iterable, List, Tuple
from urllib.parse import quote

import folium
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
import streamlit as st
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from streamlit_folium import st_folium

APP_DIR = Path(__file__).resolve().parent
DATA_PATH = APP_DIR / "cities_uzbekistan.csv"
ASSET_DIR = APP_DIR / "assets"
FEEDBACK_PATH = APP_DIR / "feedback_state.json"

FEATURE_COLUMNS = [
    "affordability",
    "grocery_affordability",
    "entertainment",
    "sunset_scenery",
    "safety",
    "jobs",
    "internet",
    "healthcare",
    "education",
    "heritage",
    "nature",
    "climate_comfort",
    "quietness",
    "mobility",
]

FEATURE_LABELS = {
    "affordability": "Affordable living",
    "grocery_affordability": "Affordable groceries",
    "entertainment": "Entertainment",
    "sunset_scenery": "Sunset & scenery",
    "safety": "Safety",
    "jobs": "Jobs & career",
    "internet": "Internet & remote work",
    "healthcare": "Healthcare",
    "education": "Education",
    "heritage": "History & culture",
    "nature": "Nature access",
    "climate_comfort": "Climate comfort",
    "quietness": "Quiet lifestyle",
    "mobility": "Transportation",
}

SYNONYMS = {
    "affordability": [
        "cheap", "cheapest", "affordable", "low cost", "low-cost", "budget",
        "inexpensive", "save money", "low rent", "reasonable rent",
    ],
    "grocery_affordability": [
        "cheap grocery", "cheap groceries", "grocery price", "food price",
        "affordable food", "low food cost", "market price", "cheap market",
    ],
    "entertainment": [
        "entertainment", "nightlife", "fun", "activities", "events", "concert",
        "cinema", "shopping", "restaurants", "social life", "things to do",
    ],
    "sunset_scenery": [
        "sunset", "sunsets", "scenic", "beautiful view", "view", "photography",
        "sky", "romantic", "landscape",
    ],
    "safety": ["safe", "safety", "secure", "low crime", "family friendly"],
    "jobs": ["job", "jobs", "career", "employment", "business", "salary", "startup"],
    "internet": [
        "internet", "wifi", "remote work", "digital nomad", "online work",
        "technology", "tech", "fast connection",
    ],
    "healthcare": ["hospital", "healthcare", "doctor", "medical", "clinic", "health"],
    "education": [
        "school", "education", "university", "college", "student", "children",
        "academic", "learning",
    ],
    "heritage": [
        "history", "historic", "heritage", "culture", "architecture", "museum",
        "old city", "silk road", "traditional",
    ],
    "nature": [
        "nature", "mountain", "mountains", "green", "park", "hiking", "outdoor",
        "river", "lake", "desert", "fresh air",
    ],
    "climate_comfort": [
        "comfortable weather", "mild climate", "climate", "weather", "not too hot",
        "not too cold", "pleasant weather",
    ],
    "quietness": ["quiet", "calm", "peaceful", "slow life", "less crowded", "relaxed"],
    "mobility": [
        "transport", "transportation", "metro", "bus", "airport", "walkable",
        "commute", "easy travel", "connected",
    ],
}

CITY_IMAGE_FILES = {
    'Tashkent': 'cities/tashkent.jpg',
    'Samarkand': 'cities/samarkand.jpg',
    'Bukhara': 'cities/bukhara.jpg',
    'Khiva': 'cities/khiva.jpg',
    'Nukus': 'cities/nukus.jpg',
    'Fergana': 'cities/fergana.jpg',
    'Andijan': 'cities/andijan.jpg',
    'Namangan': 'cities/namangan.jpg',
    'Qarshi': 'cities/qarshi.jpg',
    'Termez': 'cities/termez.jpg',
    'Jizzakh': 'cities/jizzakh.jpg',
    'Gulistan': 'cities/gulistan.jpg',
    'Navoi': 'cities/navoi.jpg',
    'Urgench': 'cities/urgench.jpg',
    'Kokand': 'cities/kokand.jpg',
    'Shahrisabz': 'cities/shahrisabz.jpg',
}
FALLBACK_CITY_IMAGE = "spot-forest.jpg"


def city_image_file(city: object) -> str:
    """Return the repository-relative image assigned to a city."""
    filename = CITY_IMAGE_FILES.get(str(city).strip(), FALLBACK_CITY_IMAGE)
    return filename if (ASSET_DIR / filename).exists() else FALLBACK_CITY_IMAGE


def city_image_path(city: object) -> Path:
    return ASSET_DIR / city_image_file(city)


def image_data_uri(filename: str) -> str:
    path = ASSET_DIR / filename
    if not path.exists():
        return ""
    mime = "image/jpeg" if path.suffix.lower() in {".jpg", ".jpeg"} else "image/png"
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:{mime};base64,{encoded}"


@st.cache_data
def load_city_data() -> pd.DataFrame:
    df = pd.read_csv(DATA_PATH)
    for column in FEATURE_COLUMNS:
        df[column] = pd.to_numeric(df[column], errors="coerce").fillna(50).clip(0, 100)
    return df


def city_document(row: pd.Series) -> str:
    feature_words = " ".join(
        FEATURE_LABELS[column] for column in FEATURE_COLUMNS if float(row[column]) >= 72
    )
    return " ".join(
        [
            str(row.get("city", "")),
            str(row.get("region", "")),
            str(row.get("tags", "")),
            str(row.get("description", "")),
            feature_words,
        ]
    )


@st.cache_resource
def build_text_index(documents: Tuple[str, ...]) -> Tuple[TfidfVectorizer, object]:
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words="english", min_df=1)
    matrix = vectorizer.fit_transform(documents)
    return vectorizer, matrix


def parse_preference_weights(query: str) -> Tuple[Dict[str, float], List[str]]:
    text = re.sub(r"\s+", " ", query.lower()).strip()
    weights = {feature: 0.0 for feature in FEATURE_COLUMNS}
    matches: List[str] = []

    for feature, phrases in SYNONYMS.items():
        for phrase in phrases:
            if phrase in text:
                weights[feature] += 1.4 if " " in phrase else 1.0
                matches.append(f"{FEATURE_LABELS[feature]} <- '{phrase}'")

    if "family" in text or "children" in text or "kids" in text:
        weights["safety"] += 1.1
        weights["education"] += 0.9
        weights["healthcare"] += 0.6
        matches.append("Family intent -> safety, education, healthcare")
    if "retire" in text or "retirement" in text:
        weights["affordability"] += 0.8
        weights["quietness"] += 1.0
        weights["healthcare"] += 0.8
        matches.append("Retirement intent -> affordability, quietness, healthcare")
    if "young professional" in text or "professional" in text:
        weights["jobs"] += 1.0
        weights["internet"] += 0.7
        weights["entertainment"] += 0.5
        matches.append("Professional intent -> jobs, internet, entertainment")
    if "tourist" in text or "tourism" in text or "visit" in text:
        weights["heritage"] += 0.9
        weights["entertainment"] += 0.5
        weights["mobility"] += 0.4
        matches.append("Tourism intent -> heritage, activities, transportation")

    total = sum(weights.values())
    if total <= 0:
        defaults = {
            "affordability": 1.0,
            "safety": 1.0,
            "healthcare": 0.8,
            "internet": 0.7,
            "mobility": 0.6,
            "entertainment": 0.5,
        }
        weights.update(defaults)
        matches.append("No strong keyword found -> balanced default profile")
        total = sum(weights.values())

    return {key: value / total for key, value in weights.items()}, matches


def combine_weights(text_weights: Dict[str, float], slider_weights: Dict[str, float]) -> Dict[str, float]:
    combined = {
        feature: text_weights.get(feature, 0.0) + slider_weights.get(feature, 0.0) / 5.0
        for feature in FEATURE_COLUMNS
    }
    total = sum(combined.values()) or 1.0
    return {feature: value / total for feature, value in combined.items()}


def load_feedback_state(cities: Iterable[str]) -> Dict[str, Dict[str, int]]:
    default = {str(city): {"likes": 0, "dislikes": 0} for city in cities}
    if not FEEDBACK_PATH.exists():
        return default
    try:
        stored = json.loads(FEEDBACK_PATH.read_text(encoding="utf-8"))
        for city in default:
            values = stored.get(city, {})
            default[city]["likes"] = int(values.get("likes", 0))
            default[city]["dislikes"] = int(values.get("dislikes", 0))
    except (OSError, ValueError, TypeError):
        pass
    return default


def save_feedback_state(state: Dict[str, Dict[str, int]]) -> None:
    try:
        FEEDBACK_PATH.write_text(json.dumps(state, indent=2), encoding="utf-8")
    except OSError:
        pass


def bandit_posterior(city: str, state: Dict[str, Dict[str, int]]) -> float:
    values = state.get(city, {"likes": 0, "dislikes": 0})
    likes = max(0, int(values.get("likes", 0)))
    dislikes = max(0, int(values.get("dislikes", 0)))
    return (likes + 1.0) / (likes + dislikes + 2.0)


def make_dnn_training_data(df: pd.DataFrame, n_samples: int = 1400) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(42)
    features = df[FEATURE_COLUMNS].to_numpy(dtype=float) / 100.0
    x_rows: List[np.ndarray] = []
    y_rows: List[int] = []

    for _ in range(n_samples):
        city_vector = features[rng.integers(0, len(features))]
        preference = rng.dirichlet(np.ones(len(FEATURE_COLUMNS)) * 0.75)
        interaction = preference * city_vector
        utility = float(np.dot(preference, city_vector))

        idx = {name: FEATURE_COLUMNS.index(name) for name in FEATURE_COLUMNS}
        utility += 0.09 * min(
            preference[idx["heritage"]] * city_vector[idx["heritage"]],
            preference[idx["entertainment"]] * city_vector[idx["entertainment"]],
        )
        utility += 0.08 * min(
            preference[idx["affordability"]] * city_vector[idx["affordability"]],
            preference[idx["grocery_affordability"]] * city_vector[idx["grocery_affordability"]],
        )
        utility += 0.07 * min(
            preference[idx["jobs"]] * city_vector[idx["jobs"]],
            preference[idx["internet"]] * city_vector[idx["internet"]],
        )
        utility += 0.06 * min(
            preference[idx["nature"]] * city_vector[idx["nature"]],
            preference[idx["sunset_scenery"]] * city_vector[idx["sunset_scenery"]],
        )
        utility += rng.normal(0, 0.025)

        x_rows.append(np.concatenate([preference, city_vector, interaction]))
        like_probability = 1.0 / (1.0 + math.exp(-(utility - 0.73) / 0.055))
        y_rows.append(int(rng.random() < like_probability))

    return np.vstack(x_rows), np.asarray(y_rows, dtype=int)


@st.cache_resource
def train_demo_dnn(data_signature: str, _df: pd.DataFrame) -> Pipeline:
    del data_signature
    x_train, y_train = make_dnn_training_data(_df)
    model = Pipeline(
        steps=[
            ("scale", StandardScaler()),
            (
                "dnn",
                MLPClassifier(
                    hidden_layer_sizes=(64, 32, 16),
                    activation="relu",
                    alpha=0.001,
                    learning_rate_init=0.002,
                    max_iter=220,
                    early_stopping=True,
                    random_state=42,
                ),
            ),
        ]
    )
    model.fit(x_train, y_train)
    return model


def dnn_like_probabilities(model: Pipeline, df: pd.DataFrame, weights: Dict[str, float]) -> np.ndarray:
    preference = np.array([weights[feature] for feature in FEATURE_COLUMNS], dtype=float)
    city_features = df[FEATURE_COLUMNS].to_numpy(dtype=float) / 100.0
    x = np.hstack(
        [
            np.repeat(preference.reshape(1, -1), len(df), axis=0),
            city_features,
            city_features * preference,
        ]
    )
    return model.predict_proba(x)[:, 1]


def rank_cities(
    df: pd.DataFrame,
    query: str,
    slider_weights: Dict[str, float],
    feedback_state: Dict[str, Dict[str, int]],
    top_n: int,
) -> Tuple[pd.DataFrame, Dict[str, float], List[str]]:
    text_weights, matches = parse_preference_weights(query)
    weights = combine_weights(text_weights, slider_weights)

    documents = tuple(city_document(row) for _, row in df.iterrows())
    vectorizer, city_matrix = build_text_index(documents)
    query_vector = vectorizer.transform([query or "balanced affordable safe city"])
    text_similarity = cosine_similarity(query_vector, city_matrix).ravel()

    feature_matrix = df[FEATURE_COLUMNS].to_numpy(dtype=float) / 100.0
    weight_vector = np.array([weights[column] for column in FEATURE_COLUMNS], dtype=float)
    preference_score = feature_matrix @ weight_vector

    signature = str(hash(tuple(np.round(feature_matrix.ravel(), 4))))
    dnn_model = train_demo_dnn(signature, df)
    dnn_score = dnn_like_probabilities(dnn_model, df, weights)
    posterior = np.array([bandit_posterior(city, feedback_state) for city in df["city"]], dtype=float)

    final_score = 0.68 * preference_score + 0.14 * text_similarity + 0.13 * dnn_score + 0.05 * posterior

    ranked = df.copy()
    ranked["preference_score"] = preference_score * 100
    ranked["text_similarity"] = text_similarity * 100
    ranked["dnn_like_probability"] = dnn_score * 100
    ranked["feedback_posterior"] = posterior * 100
    ranked["match_score"] = final_score * 100
    ranked = ranked.sort_values("match_score", ascending=False).head(top_n).reset_index(drop=True)
    return ranked, weights, matches


def top_reasons(row: pd.Series, weights: Dict[str, float], n: int = 4) -> List[str]:
    contributions = []
    for feature in FEATURE_COLUMNS:
        contributions.append((weights.get(feature, 0.0) * float(row[feature]), feature, float(row[feature])))
    contributions.sort(reverse=True)
    return [f"{FEATURE_LABELS[feature]}: {value:.0f}/100" for _, feature, value in contributions[:n]]


def cluster_cities(df: pd.DataFrame) -> pd.DataFrame:
    output = df.copy()
    model = KMeans(n_clusters=4, random_state=42, n_init=20)
    output["cluster_id"] = model.fit_predict(df[FEATURE_COLUMNS])
    summaries = output.groupby("cluster_id")[FEATURE_COLUMNS].mean()
    labels: Dict[int, str] = {}
    for cluster_id, values in summaries.iterrows():
        if values["jobs"] + values["healthcare"] + values["education"] >= 230:
            label = "Career and services hub"
        elif values["heritage"] + values["entertainment"] >= 145:
            label = "Culture and tourism hub"
        elif values["affordability"] + values["quietness"] >= 155:
            label = "Affordable quiet city"
        else:
            label = "Balanced regional center"
        labels[int(cluster_id)] = label
    output["city_archetype"] = output["cluster_id"].map(labels)
    return output


def build_map(df: pd.DataFrame, ranked: pd.DataFrame | None = None) -> folium.Map:
    map_object = folium.Map(location=[41.2, 64.6], zoom_start=5, tiles="OpenStreetMap", control_scale=True)
    ranked_lookup = {}
    if ranked is not None:
        ranked_lookup = {
            city: (index + 1, float(score))
            for index, (city, score) in enumerate(zip(ranked["city"], ranked["match_score"]))
        }

    for _, row in df.iterrows():
        city = str(row["city"])
        rank_text = ""
        icon_color = "darkgreen"
        if city in ranked_lookup:
            rank, score = ranked_lookup[city]
            rank_text = f"<br><b>Recommendation rank:</b> #{rank}<br><b>Match:</b> {score:.1f}/100"
            icon_color = "green" if rank == 1 else "cadetblue"
        popup = folium.Popup(
            f"<b>{city}</b><br>{row['region']}<br>{row['description']}{rank_text}",
            max_width=330,
        )
        folium.Marker(
            location=[float(row["lat"]), float(row["lon"])],
            tooltip=city,
            popup=popup,
            icon=folium.Icon(color=icon_color, icon="home"),
        ).add_to(map_object)
    return map_object


@st.cache_data(ttl=1800, show_spinner=False)
def fetch_live_weather(lat: float, lon: float) -> Dict[str, object]:
    response = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lat,
            "longitude": lon,
            "current": "temperature_2m,apparent_temperature,weather_code,wind_speed_10m",
            "daily": "temperature_2m_max,temperature_2m_min,sunset",
            "forecast_days": 3,
            "timezone": "auto",
        },
        timeout=7,
    )
    response.raise_for_status()
    return response.json()


@st.cache_data(ttl=86400, show_spinner=False)
def fetch_wikipedia_summary(city: str) -> Dict[str, object]:
    response = requests.get(
        f"https://en.wikipedia.org/api/rest_v1/page/summary/{quote(city)}",
        timeout=7,
        headers={"User-Agent": "AkbarxonLivingAdvisorPrototype/2.0"},
    )
    response.raise_for_status()
    return response.json()


def weather_code_label(code: int | float | None) -> str:
    if code is None:
        return "Unknown"
    code = int(code)
    if code == 0:
        return "Clear sky"
    if code in {1, 2, 3}:
        return "Partly cloudy"
    if code in {45, 48}:
        return "Fog"
    if 51 <= code <= 67:
        return "Rain or drizzle"
    if 71 <= code <= 77:
        return "Snow"
    if 80 <= code <= 82:
        return "Rain showers"
    if code >= 95:
        return "Thunderstorm"
    return "Mixed conditions"


def radar_figure(row: pd.Series, features: List[str]) -> go.Figure:
    values = [float(row[feature]) for feature in features]
    labels = [FEATURE_LABELS[feature] for feature in features]
    fig = go.Figure(
        data=[go.Scatterpolar(r=values + [values[0]], theta=labels + [labels[0]], fill="toself", name=str(row["city"]))]
    )
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
        showlegend=False,
        margin=dict(l=28, r=28, t=28, b=28),
        height=420,
    )
    return fig


def inject_css() -> None:
    st.markdown(
        """
        <style>
        :root {
          --bg: #fcfcfc;
          --fg: #333333;
          --muted: #7c7c7c;
          --primary: #8ea89b;
          --primary-dark: #536f61;
          --accent: #eef5f1;
          --border: #e9e9e9;
          --card: #ffffff;
        }
        html {scroll-behavior: smooth;}
        .stApp {background: var(--bg); color: var(--fg);}

        /*
         * Streamlit Community Cloud keeps its native toolbar fixed at the top.
         * The previous 1rem top padding pulled the custom brand/navigation under
         * that toolbar. Reserve a safe top zone for the main canvas instead.
         */
        .block-container {
          max-width: 1280px;
          padding-top: 4.75rem !important;
          padding-bottom: 4rem;
        }
        [data-testid="stMainBlockContainer"] {
          max-width: 1280px;
          padding-top: 4.75rem !important;
          padding-bottom: 4rem;
        }
        h1, h2, h3, h4 {letter-spacing: -0.02em;}
        [data-testid="stSidebar"] {background: #f9faf9; border-right: 1px solid var(--border);}
        [data-testid="stSidebar"] .block-container {padding-top: 1rem !important;}
        [data-testid="stSidebar"] [data-testid="stSidebarContent"] {padding-top: .35rem;}

        .top-brand {
          display:flex;
          align-items:center;
          justify-content:space-between;
          gap:1rem;
          min-height:42px;
          padding:.35rem .15rem 1rem .15rem;
          position:relative;
          z-index:1;
        }
        .brand-left {display:flex; align-items:center; gap:.7rem;}
        .brand-mark {width:34px; height:34px; border-radius:50%; display:grid; place-items:center; border:1px solid var(--border); background:white; font-size:18px;}
        .brand-name {font-size:1.02rem; font-weight:500; letter-spacing:.01em;}
        .team-mini {font-size:.78rem; color:var(--muted); text-align:right; line-height:1.45;}

        .hero-shell {position:relative; min-height:565px; border-radius:18px; overflow:hidden; margin: .4rem 0 4rem 0; box-shadow: 0 12px 38px rgba(0,0,0,.08);}
        .hero-shell img {width:100%; height:565px; object-fit:cover; display:block;}
        .hero-overlay {position:absolute; inset:0; background:linear-gradient(180deg, rgba(0,0,0,.05) 0%, rgba(0,0,0,.42) 100%);}
        .hero-copy {position:absolute; left:42px; bottom:54px; color:white; max-width:560px;}
        .hero-eyebrow {font-size:.74rem; text-transform:uppercase; letter-spacing:.18em; margin-bottom:1rem; opacity:.92;}
        .hero-title {font-size:clamp(2.45rem,5vw,4.7rem); font-weight:300; line-height:.96; margin:0 0 1.2rem 0;}
        .hero-sub {font-size:1rem; line-height:1.65; max-width:520px; opacity:.94; margin-bottom:1.4rem;}
        .hero-pill {display:inline-block; background:#fff; color:#303030; padding:.8rem 1.15rem; border-radius:999px; font-size:.82rem; text-decoration:none;}
        .hero-bars {position:absolute; bottom:22px; left:42px; right:42px; display:flex; gap:8px;}
        .hero-bars span {height:2px; flex:1; background:rgba(255,255,255,.35);}
        .hero-bars span:first-child {background:white;}

        .section-head {text-align:center; margin: 1.5rem 0 2.5rem 0;}
        .eyebrow {font-size:.7rem; text-transform:uppercase; letter-spacing:.16em; color:var(--muted); margin-bottom:.7rem;}
        .section-title {font-size:2rem; font-weight:300; margin:0 0 .7rem 0;}
        .section-sub {font-size:.93rem; color:var(--muted); max-width:620px; margin:0 auto; line-height:1.6;}

        .city-card-html {overflow:hidden; border:1px solid var(--border); background:var(--card); border-radius:12px; box-shadow:0 8px 24px rgba(0,0,0,.055); min-height:100%; height:100%; margin-bottom:1.1rem;}
        .city-card-html img {width:100%; height:205px; object-fit:cover; display:block;}
        .city-card-body {padding:1.25rem 1.25rem 1.35rem 1.25rem;}
        .city-card-top {display:flex; justify-content:space-between; gap:1rem; align-items:flex-start;}
        .city-card-title {font-size:1.05rem; font-weight:500; margin:0;}
        .city-card-score {font-size:.76rem; padding:.3rem .55rem; border-radius:999px; background:var(--accent); color:var(--primary-dark); white-space:nowrap;}
        .city-region {font-size:.78rem; color:var(--muted); margin:.3rem 0 .85rem 0;}
        .city-desc {font-size:.86rem; color:#5e5e5e; line-height:1.55; min-height:5.35em; overflow:hidden; display:-webkit-box; -webkit-line-clamp:4; -webkit-box-orient:vertical;}
        .chips {display:flex; flex-wrap:wrap; gap:6px; margin-top:.8rem;}
        .chip {font-size:.64rem; text-transform:uppercase; letter-spacing:.08em; background:var(--accent); padding:.3rem .45rem; border-radius:4px; color:#4f655a;}

        .experience-row {display:flex; align-items:center; gap:1rem; padding:1.2rem 1.25rem; border:1px solid rgba(255,255,255,.8); background:rgba(255,255,255,.72); border-radius:10px; margin:.75rem 0; box-shadow:0 2px 12px rgba(0,0,0,.035); transition:.25s ease;}
        .experience-row:hover {transform:translateY(-3px); box-shadow:0 10px 25px rgba(0,0,0,.07);}
        .experience-icon {width:46px; height:46px; border-radius:50%; display:grid; place-items:center; background:var(--accent); font-size:21px; flex:0 0 auto;}
        .experience-title {font-size:.92rem; font-weight:500; margin-bottom:.25rem;}
        .experience-desc {font-size:.82rem; color:var(--muted); line-height:1.5;}

        .advisor-card {border:1px solid var(--border); border-radius:14px; background:white; padding:1.5rem; box-shadow:0 8px 26px rgba(0,0,0,.045);}
        .result-card {border:1px solid var(--border); border-radius:12px; background:white; padding:1.25rem; margin:.75rem 0;}
        .result-rank {font-size:.72rem; text-transform:uppercase; letter-spacing:.12em; color:var(--muted);}
        .result-title {font-size:1.3rem; font-weight:400; margin:.3rem 0;}
        .score-badge {display:inline-block; background:var(--accent); color:var(--primary-dark); border-radius:999px; padding:.35rem .65rem; font-size:.78rem; font-weight:600;}
        .result-note {font-size:.78rem; color:var(--muted); line-height:1.55;}

        .team-card {padding:1.05rem 1rem; border:1px solid var(--border); border-radius:12px; background:white; margin-bottom:1rem;}
        .team-title {font-size:.85rem; text-transform:uppercase; letter-spacing:.1em; color:var(--muted); margin-bottom:.65rem;}
        .team-name {font-size:.9rem; line-height:1.65;}
        .prototype-note {font-size:.8rem; color:#666; line-height:1.55; padding:.85rem; background:#f1f6f3; border-radius:10px; border:1px solid #e3eee8;}

        .ai-card {padding:1.3rem; border:1px solid var(--border); border-radius:12px; background:white; min-height:180px;}
        .ai-num {font-size:.7rem; letter-spacing:.14em; color:var(--muted); text-transform:uppercase;}
        .ai-title {font-size:1rem; font-weight:500; margin:.45rem 0;}
        .ai-desc {font-size:.84rem; line-height:1.58; color:#6b6b6b;}

        .footer-shell {background:#2f302f; color:#f6f6f6; border-radius:14px; padding:2rem 2.2rem; margin-top:4rem;}
        .footer-brand {font-size:1rem; font-weight:500; margin-bottom:.5rem;}
        .footer-copy {font-size:.8rem; color:rgba(255,255,255,.68); line-height:1.6;}
        .footer-rule {border-top:1px solid rgba(255,255,255,.16); margin:1.4rem 0 1rem 0;}

        div[data-testid="stRadio"] > div {gap:.15rem;}
        div[data-testid="stRadio"] label {border-radius:999px; padding:.18rem .55rem;}
        div[data-testid="stButton"] button {border-radius:999px; font-weight:500;}
        div[data-testid="stFormSubmitButton"] button {border-radius:999px;}
        .stTextArea textarea, .stSelectbox div[data-baseweb="select"] > div {border-radius:10px;}

        @media (max-width: 800px) {
          .hero-shell, .hero-shell img {min-height:470px; height:470px;}
          .hero-copy {left:24px; right:24px; bottom:46px;}
          .hero-bars {left:24px; right:24px;}
          .team-mini {display:none;}
          .hero-title {font-size:2.7rem;}
        }
        </style>
        """,
        unsafe_allow_html=True,
    )


def render_top_brand() -> None:
    st.markdown(
        """
        <div class="top-brand">
          <div class="brand-left">
            <div class="brand-mark">🧭</div>
            <div class="brand-name">Akbarxon AI Living Advisor</div>
          </div>
          <div class="team-mini">Author: Akbarxon Nasirov<br>Mentor: Dr. Qingyang Xiao</div>
        </div>
        """,
        unsafe_allow_html=True,
    )


def render_hero() -> None:
    hero = image_data_uri("hero-camping.jpg")
    st.markdown(
        f"""
        <div class="hero-shell">
          <img src="{hero}" alt="Lifestyle landscape design visual">
          <div class="hero-overlay"></div>
          <div class="hero-copy">
            <div class="hero-eyebrow">AI-powered city discovery · Uzbekistan</div>
            <div class="hero-title">Find a place<br>that fits your life</div>
            <div class="hero-sub">Describe the lifestyle you want. The advisor translates your words into priorities, ranks Uzbekistan cities, maps the results, and learns from feedback.</div>
            <a class="hero-pill" href="#advisor">Start Exploring &nbsp;→</a>
          </div>
          <div class="hero-bars"><span></span><span></span><span></span><span></span></div>
        </div>
        """,
        unsafe_allow_html=True,
    )


def render_section_head(eyebrow: str, title: str, subtitle: str) -> None:
    st.markdown(
        f"""
        <div class="section-head">
          <div class="eyebrow">{eyebrow}</div>
          <div class="section-title">{title}</div>
          <div class="section-sub">{subtitle}</div>
        </div>
        """,
        unsafe_allow_html=True,
    )


def build_city_card_html(row: pd.Series, rank: int | None = None) -> str:
    """Build one self-contained card without Markdown blank-line parsing issues."""
    city = html.escape(str(row.get("city", "Unknown city")))
    region = html.escape(str(row.get("region", "")))
    description = html.escape(str(row.get("description", "")))
    image = html.escape(image_data_uri(city_image_file(row.get("city", ""))), quote=True)
    tags = [item.strip() for item in str(row.get("tags", "")).split(",") if item.strip()][:3]
    chips = "".join(f'<span class="chip">{html.escape(tag)}</span>' for tag in tags)

    raw_score = row.get("match_score", 0.0)
    try:
        score = float(raw_score)
    except (TypeError, ValueError):
        score = 0.0
    score_html = (
        f'<span class="city-card-score">{score:.1f} match</span>'
        if math.isfinite(score) and score > 0
        else ""
    )
    rank_text = f"#{rank} · " if rank else ""

    # Join the HTML fragments directly. A blank line inside a raw Markdown HTML
    # block can terminate the block and expose closing tags as visible text.
    return "".join(
        [
            '<div class="city-card-html">',
            f'<img src="{image}" alt="{city} city photo" loading="lazy">',
            '<div class="city-card-body">',
            '<div class="city-card-top">',
            f'<div class="city-card-title">{rank_text}{city}</div>',
            score_html,
            '</div>',
            f'<div class="city-region">&#128205; {region}</div>',
            f'<div class="city-desc">{description}</div>',
            f'<div class="chips">{chips}</div>',
            '</div>',
            '</div>',
        ]
    )


def render_city_card(row: pd.Series, rank: int | None = None) -> None:
    # st.html renders the card as HTML rather than asking the Markdown parser to
    # infer where the raw HTML block begins and ends.
    st.html(build_city_card_html(row, rank=rank))


def render_experience_rows() -> None:
    items = [
        ("🧠", "Natural-language understanding", "Turn everyday phrases such as 'cheap groceries' or 'great sunsets' into measurable lifestyle priorities."),
        ("🗺️", "Map-first exploration", "See candidate cities geographically and inspect how recommendations relate across Uzbekistan."),
        ("📊", "Transparent scoring", "Combine preference features, text similarity, neural-network estimates, and feedback into an explainable score."),
        ("👍", "Feedback learning", "Likes and dislikes update a lightweight bandit signal so the prototype can adapt over time."),
    ]
    for icon, title, desc in items:
        st.markdown(
            f"""
            <div class="experience-row">
              <div class="experience-icon">{icon}</div>
              <div><div class="experience-title">{title}</div><div class="experience-desc">{desc}</div></div>
            </div>
            """,
            unsafe_allow_html=True,
        )


def render_feedback_buttons(city: str, feedback_state: Dict[str, Dict[str, int]], key_prefix: str) -> None:
    left, right, stats = st.columns([1, 1, 2.4])
    if left.button("👍 Like", key=f"{key_prefix}_like_{city}", use_container_width=True):
        feedback_state[city]["likes"] += 1
        save_feedback_state(feedback_state)
        st.toast(f"Feedback saved for {city}")
        st.rerun()
    if right.button("👎 Not for me", key=f"{key_prefix}_dislike_{city}", use_container_width=True):
        feedback_state[city]["dislikes"] += 1
        save_feedback_state(feedback_state)
        st.toast(f"Feedback saved for {city}")
        st.rerun()
    values = feedback_state[city]
    stats.caption(
        f"Prototype feedback: {values['likes']} likes · {values['dislikes']} dislikes · "
        f"bandit confidence {bandit_posterior(city, feedback_state):.2f}"
    )


def ensure_default_results(
    df: pd.DataFrame,
    slider_weights: Dict[str, float],
    feedback_state: Dict[str, Dict[str, int]],
    top_n: int,
) -> None:
    if "ranked_results" not in st.session_state:
        query = "I want an affordable city with cheap groceries, beautiful sunsets, and good entertainment."
        ranked, weights, matches = rank_cities(df, query, slider_weights, feedback_state, top_n)
        st.session_state["ranked_results"] = ranked
        st.session_state["active_weights"] = weights
        st.session_state["query_matches"] = matches
        st.session_state["active_query"] = query


def render_home(
    df: pd.DataFrame,
    slider_weights: Dict[str, float],
    feedback_state: Dict[str, Dict[str, int]],
    top_n: int,
) -> None:
    ensure_default_results(df, slider_weights, feedback_state, top_n)
    render_hero()

    render_section_head(
        "Featured matches",
        "Lifestyle ideas, ranked for you",
        "A first look at cities selected by the same AI engine used in the full recommendation workspace.",
    )
    ranked = st.session_state["ranked_results"].head(3)
    cols = st.columns(3)
    for idx, (col, (_, row)) in enumerate(zip(cols, ranked.iterrows())):
        with col:
            render_city_card(row, rank=idx + 1)

    st.markdown("<div style='height:3rem'></div>", unsafe_allow_html=True)
    render_section_head(
        "The experience",
        "Minimal design, useful intelligence",
        "The uploaded UI's calm, image-led design language is preserved while the interaction is rebuilt around city discovery.",
    )
    render_experience_rows()

    st.markdown('<div id="advisor"></div>', unsafe_allow_html=True)
    st.markdown("<div style='height:2rem'></div>", unsafe_allow_html=True)
    render_section_head(
        "AI advisor",
        "Describe your ideal place",
        "Use natural language now; detailed controls are available in the sidebar and the Recommend workspace.",
    )
    with st.form("home_quick_advisor"):
        quick_query = st.text_area(
            "Lifestyle request",
            value=st.session_state.get("active_query", "I want an affordable city with good food prices and beautiful scenery."),
            height=115,
            label_visibility="collapsed",
        )
        submitted = st.form_submit_button("Find my cities", type="primary", use_container_width=True)
    if submitted:
        ranked, weights, matches = rank_cities(df, quick_query, slider_weights, feedback_state, top_n)
        st.session_state["ranked_results"] = ranked
        st.session_state["active_weights"] = weights
        st.session_state["query_matches"] = matches
        st.session_state["active_query"] = quick_query
        st.success("Recommendations updated. Open the Recommend or Map & Compare workspace for full details.")
        st.dataframe(ranked[["city", "region", "match_score"]].head(5), hide_index=True, use_container_width=True)


def render_recommend(
    df: pd.DataFrame,
    slider_weights: Dict[str, float],
    feedback_state: Dict[str, Dict[str, int]],
    top_n: int,
) -> None:
    render_section_head(
        "Personalized advisor",
        "Tell the AI what matters",
        "Text is the main signal. The ranking engine combines interpretable lifestyle features, NLP similarity, a neural model, and feedback.",
    )

    query = st.text_area(
        "What kind of place are you looking for?",
        value=st.session_state.get(
            "active_query",
            "I want an affordable city with cheap groceries, beautiful sunsets, and good entertainment.",
        ),
        height=120,
        help="Examples: family-friendly and safe; best for remote work; historic and walkable; quiet retirement city.",
    )
    if st.button("Find my best cities", type="primary", use_container_width=True):
        ranked, weights, matches = rank_cities(df, query, slider_weights, feedback_state, top_n)
        st.session_state["ranked_results"] = ranked
        st.session_state["active_weights"] = weights
        st.session_state["query_matches"] = matches
        st.session_state["active_query"] = query

    ensure_default_results(df, slider_weights, feedback_state, top_n)
    ranked = st.session_state["ranked_results"]
    weights = st.session_state["active_weights"]
    matches = st.session_state["query_matches"]

    with st.expander("How the AI interpreted your request", expanded=False):
        st.write("Detected signals:")
        for match in matches:
            st.write(f"- {match}")
        weight_table = pd.DataFrame(
            {
                "Preference": [FEATURE_LABELS[key] for key in FEATURE_COLUMNS],
                "Weight (%)": [round(weights[key] * 100, 1) for key in FEATURE_COLUMNS],
            }
        ).sort_values("Weight (%)", ascending=False)
        st.dataframe(weight_table, hide_index=True, use_container_width=True)

    st.markdown("### Top matches")
    for index, row in ranked.iterrows():
        reasons = top_reasons(row, weights)
        image_path = city_image_path(row["city"])
        left, right = st.columns([1, 2.2])
        with left:
            st.image(image_path, use_container_width=True, caption=f"{row['city']} city photo")
        with right:
            st.markdown(
                f"""
                <div class="result-card">
                  <div class="result-rank">Recommendation #{index + 1}</div>
                  <div class="result-title">{row['city']} &nbsp; <span class="score-badge">{row['match_score']:.1f}/100 match</span></div>
                  <div class="city-region">{row['region']}</div>
                  <p>{row['description']}</p>
                  <p><b>Why it fits:</b> {' · '.join(reasons)}</p>
                  <div class="result-note">Hybrid components: preference {row['preference_score']:.1f}, text {row['text_similarity']:.1f}, neural model {row['dnn_like_probability']:.1f}, feedback {row['feedback_posterior']:.1f}</div>
                </div>
                """,
                unsafe_allow_html=True,
            )
            render_feedback_buttons(str(row["city"]), feedback_state, f"rec_{index}")

    st.markdown("### Optional live context")
    selected_live_city = st.selectbox("City", ranked["city"].tolist(), key="live_city")
    if st.button("Fetch weather + city summary"):
        city_row = df.loc[df["city"] == selected_live_city].iloc[0]
        with st.spinner("Retrieving public context..."):
            try:
                wiki = fetch_wikipedia_summary(selected_live_city)
                weather = fetch_live_weather(float(city_row["lat"]), float(city_row["lon"]))
                current = weather.get("current", {})
                c1, c2, c3 = st.columns(3)
                c1.metric("Temperature", f"{current.get('temperature_2m', '—')} °C")
                c2.metric("Feels like", f"{current.get('apparent_temperature', '—')} °C")
                c3.metric("Conditions", weather_code_label(current.get("weather_code")))
                st.write(wiki.get("extract", "No summary was returned."))
                source_url = wiki.get("content_urls", {}).get("desktop", {}).get("page")
                if source_url:
                    st.link_button("Open source article", source_url)
            except requests.RequestException as exc:
                st.warning(f"Live data is temporarily unavailable: {exc}")


def render_map_compare(df: pd.DataFrame, slider_weights, feedback_state, top_n) -> None:
    ensure_default_results(df, slider_weights, feedback_state, top_n)
    ranked = st.session_state["ranked_results"]
    render_section_head(
        "Map & compare",
        "See the recommendations geographically",
        "The map uses OpenStreetMap tiles and overlays the prototype Uzbekistan city profiles and current recommendation ranking.",
    )
    st_folium(build_map(df, ranked), width=None, height=590, returned_objects=[])

    st.markdown("### Recommendation score comparison")
    chart_df = ranked[["city", "match_score"]].sort_values("match_score")
    fig = px.bar(chart_df, x="match_score", y="city", orientation="h", range_x=[0, 100])
    fig.update_layout(height=390, margin=dict(l=20, r=20, t=20, b=20), xaxis_title="Match score", yaxis_title="")
    st.plotly_chart(fig, use_container_width=True)

    compare_names = st.multiselect(
        "Choose up to three cities for detailed comparison",
        df["city"].tolist(),
        default=ranked["city"].head(2).tolist(),
        max_selections=3,
    )
    comparison_features = [
        "affordability", "grocery_affordability", "entertainment", "sunset_scenery",
        "safety", "jobs", "internet", "healthcare", "heritage", "nature",
    ]
    columns = st.columns(max(1, len(compare_names)))
    for column, city in zip(columns, compare_names):
        row = df.loc[df["city"] == city].iloc[0]
        with column:
            st.plotly_chart(radar_figure(row, comparison_features), use_container_width=True)
            st.caption(row["description"])


def render_cities(df: pd.DataFrame, feedback_state: Dict[str, Dict[str, int]]) -> None:
    render_section_head(
        "City explorer",
        "Browse every prototype profile",
        "Each profile now uses the city-specific photo supplied for that named city, while preserving the uploaded location-card design.",
    )

    search = st.text_input("Filter cities", placeholder="Search by city, region, or tag")
    filtered = df.copy()
    if search.strip():
        needle = search.lower().strip()
        mask = filtered.apply(
            lambda row: needle in " ".join([str(row.get("city", "")), str(row.get("region", "")), str(row.get("tags", ""))]).lower(),
            axis=1,
        )
        filtered = filtered.loc[mask]

    for start in range(0, len(filtered), 3):
        cols = st.columns(3)
        batch = filtered.iloc[start : start + 3]
        for offset, (col, (_, row)) in enumerate(zip(cols, batch.iterrows())):
            with col:
                render_city_card(row)

    st.markdown("### Detailed city profile")
    city = st.selectbox("Select a city", df["city"].tolist(), key="city_explorer")
    row = df.loc[df["city"] == city].iloc[0]
    left, right = st.columns([1.15, 1])
    with left:
        st.image(city_image_path(city), use_container_width=True, caption=f"{city} city photo")
        st.markdown(f"## {row['city']}")
        st.write(f"**Region:** {row['region']}")
        st.write(row["description"])
        st.write(f"**Tags:** {row['tags']}")
        score_table = pd.DataFrame(
            {
                "Dimension": [FEATURE_LABELS[feature] for feature in FEATURE_COLUMNS],
                "Prototype score": [float(row[feature]) for feature in FEATURE_COLUMNS],
            }
        ).sort_values("Prototype score", ascending=False)
        st.dataframe(score_table, hide_index=True, use_container_width=True)
    with right:
        st.plotly_chart(
            radar_figure(row, ["affordability", "entertainment", "sunset_scenery", "safety", "jobs", "internet", "healthcare", "heritage", "nature"]),
            use_container_width=True,
        )
    render_feedback_buttons(city, feedback_state, "explorer")


def render_ai_lab(df: pd.DataFrame) -> None:
    render_section_head(
        "AI laboratory",
        "How the recommendation brain works",
        "The prototype intentionally combines explainable rules with machine learning, a neural-network demonstration, and feedback learning.",
    )
    c1, c2, c3, c4 = st.columns(4)
    items = [
        ("01", "Preference parser", "Phrase rules convert natural-language requests into transparent feature weights."),
        ("02", "Machine learning", "K-means groups city profiles into lifestyle archetypes from their feature vectors."),
        ("03", "Deep neural network", "A 64-32-16 MLP estimates user-city like probability using synthetic interactions for the demo."),
        ("04", "Feedback learning", "A Beta-Bernoulli bandit turns likes and dislikes into a small adaptive ranking signal."),
    ]
    for col, (num, title, desc) in zip([c1, c2, c3, c4], items):
        with col:
            st.markdown(
                f'<div class="ai-card"><div class="ai-num">{num}</div><div class="ai-title">{title}</div><div class="ai-desc">{desc}</div></div>',
                unsafe_allow_html=True,
            )

    st.markdown("### Data-driven city archetypes")
    clustered = cluster_cities(df)
    st.dataframe(clustered[["city", "region", "city_archetype"] + FEATURE_COLUMNS], hide_index=True, use_container_width=True)

    st.markdown("### Project team")
    st.markdown(
        """
        <div class="advisor-card">
          <b>Author:</b> Akbarxon Nasirov<br>
          <b>Mentor:</b> Dr. Qingyang Xiao<br><br>
          This project is designed as an educational AI portfolio platform that demonstrates data engineering, machine learning, neural networks, feedback learning, geospatial visualization, and Streamlit deployment.
        </div>
        """,
        unsafe_allow_html=True,
    )
    st.warning(
        "For production, replace illustrative scores and synthetic training data with licensed, dated, auditable sources. "
        "Use a real database for feedback, user accounts, privacy controls, source citations, and model monitoring."
    )


def render_about() -> None:
    render_section_head(
        "About the project",
        "Living decisions, made easier to explore",
        "A portfolio-ready prototype that turns broad lifestyle preferences into a structured, visual city-comparison workflow.",
    )
    image = image_data_uri("detail-forest-2.jpg")
    st.markdown(
        f"""
        <div class="hero-shell" style="min-height:430px;">
          <img src="{image}" alt="Lifestyle design visual" style="height:430px;">
          <div class="hero-overlay"></div>
          <div class="hero-copy" style="bottom:38px;">
            <div class="hero-eyebrow">Design + data + AI</div>
            <div class="hero-title" style="font-size:3rem;">Explore before<br>you relocate</div>
            <div class="hero-sub">The experience is inspired by the uploaded UI package and rebuilt in Streamlit so it can deploy directly from GitHub to Streamlit Community Cloud.</div>
          </div>
        </div>
        """,
        unsafe_allow_html=True,
    )
    left, right = st.columns(2)
    with left:
        st.markdown("### What is included")
        st.markdown(
            """
            - Natural-language city preference search
            - Uzbekistan map visualization
            - Multi-city scoring and radar comparison
            - ML clustering and neural-network demonstration
            - Like/dislike feedback learning
            - Optional public weather and city context
            - Responsive Streamlit UI styled after the uploaded React/Tailwind design
            """
        )
    with right:
        st.markdown("### UI source preservation")
        st.write(
            "The GitHub repository also contains the uploaded React/Tailwind UI project under `ui_source/`. "
            "Its `.env` file is intentionally excluded so secrets or environment-specific values are not published."
        )
        st.info(
            "The bundled photos are UI design assets. They are used as lifestyle visuals and should not be treated as verified photographs of specific Uzbekistan cities."
        )


def render_footer() -> None:
    st.markdown(
        """
        <div class="footer-shell">
          <div class="footer-brand">🧭 Akbarxon AI Living Advisor</div>
          <div class="footer-copy">AI-assisted city discovery for Uzbekistan · Streamlit portfolio prototype<br>Author: Akbarxon Nasirov · Mentor: Dr. Qingyang Xiao</div>
          <div class="footer-rule"></div>
          <div class="footer-copy">Prototype city scores are illustrative, not official statistics. Verify housing, employment, safety, healthcare, visa, legal, and financial information before making relocation decisions.</div>
        </div>
        """,
        unsafe_allow_html=True,
    )


def main() -> None:
    st.set_page_config(
        page_title="Akbarxon AI Living Advisor",
        page_icon="🧭",
        layout="wide",
        initial_sidebar_state="expanded",
    )
    inject_css()
    df = load_city_data()
    feedback_state = load_feedback_state(df["city"])

    with st.sidebar:
        st.markdown(
            """
            <div class="team-card">
              <div class="team-title">Akbarxon AI Living Advisor</div>
              <div class="team-name"><b>Author:</b> Akbarxon Nasirov<br><b>Mentor:</b> Dr. Qingyang Xiao</div>
            </div>
            """,
            unsafe_allow_html=True,
        )
        st.markdown("#### Your priorities")
        st.caption("Text is the main input. Sliders let you emphasize or correct specific priorities.")
        top_n = st.slider("Number of recommendations", 3, 8, 5)
        with st.expander("Advanced preference sliders", expanded=False):
            slider_weights = {
                feature: float(st.slider(FEATURE_LABELS[feature], 0, 5, 0, key=f"slider_{feature}"))
                for feature in FEATURE_COLUMNS
            }
        st.divider()
        st.markdown(
            """
            <div class="prototype-note"><b>Prototype note</b><br>City scores are illustrative, not official statistics. Verify housing, employment, safety, healthcare, visa, and legal information before relocating.</div>
            """,
            unsafe_allow_html=True,
        )

    render_top_brand()
    page = st.radio(
        "Navigation",
        ["Home", "Recommend", "Map & Compare", "Cities", "AI Lab", "About"],
        horizontal=True,
        label_visibility="collapsed",
    )
    st.divider()

    if page == "Home":
        render_home(df, slider_weights, feedback_state, top_n)
    elif page == "Recommend":
        render_recommend(df, slider_weights, feedback_state, top_n)
    elif page == "Map & Compare":
        render_map_compare(df, slider_weights, feedback_state, top_n)
    elif page == "Cities":
        render_cities(df, feedback_state)
    elif page == "AI Lab":
        render_ai_lab(df)
    else:
        render_about()

    render_footer()


if __name__ == "__main__":
    main()


## 4. Create GitHub and deployment files

In [ ]:
%%writefile requirements.txt
streamlit>=1.45,<2.0
pandas>=2.2,<3.0
numpy>=2.0,<3.0
scikit-learn>=1.6,<2.0
plotly>=6.0,<7.0
folium>=0.19,<1.0
streamlit-folium>=0.24,<1.0
requests>=2.32,<3.0


In [ ]:
%%writefile README.md
# Akbarxon AI Living Advisor — Uzbekistan

**Author:** Akbarxon Nasirov  
**Mentor:** Dr. Qingyang Xiao

An AI-based living-advisor platform that helps users explore which Uzbekistan cities may fit their lifestyle preferences. Users can describe what matters in natural language — affordability, grocery prices, scenery, entertainment, safety, careers, healthcare, internet, education, nature, transportation, and more — and receive ranked city recommendations with map-based visualization.

## New UI integration

This version ports the complete visual language of the supplied React/Tailwind UI package into a Streamlit-native interface:

- full-width image hero
- light minimalist design system
- muted green accents
- featured city cards
- image-based city detail layouts
- feature/experience rows
- pill-style buttons and navigation
- dark footer
- responsive layouts

The original UI project is also retained in `ui_source/` for reference. Its `.env` file is intentionally not included.

See [`UI_MIGRATION.md`](UI_MIGRATION.md) for the design mapping.


## City-photo and card-rendering repair

- Every city card, featured match, recommendation result, and detailed city profile now resolves its photo by the exact city name.
- The 16 mapped images are stored under `assets/cities/` and are included in the ZIP.
- City-card HTML is rendered with `st.html()` and compact, escaped markup. This prevents Streamlit/Markdown from exposing closing tags such as `</div>` inside the visible city description area.
- The city-specific photo is resolved by exact city name at render time, with a fallback asset if a file is ever removed.

## AI / data features

1. **Natural-language preference parsing** turns everyday requests into transparent lifestyle weights.
2. **TF-IDF + cosine similarity** compares user language against city descriptions and tags.
3. **Machine learning** uses K-means to organize prototype cities into lifestyle archetypes.
4. **Deep neural-network demonstration** uses a 64 → 32 → 16 MLP to estimate a user-city like probability from synthetic demonstration interactions.
5. **Feedback learning** uses a Beta-Bernoulli bandit signal from likes/dislikes.
6. **Interactive mapping** uses Folium/OpenStreetMap.
7. **Optional live context** can retrieve current weather from Open-Meteo and city summaries from Wikipedia when network access is available.

## Repository structure

```text
.
├── app.py
├── cities_uzbekistan.csv
├── requirements.txt
├── README.md
├── UI_MIGRATION.md
├── CITY_IMAGE_MAPPING.md
├── CHANGELOG.md
├── .streamlit/
│   └── config.toml
├── assets/
│   ├── cities/                # city-specific photos supplied for all 16 profiles
│   └── original UI design assets
├── ui_source/
│   └── original uploaded React/Tailwind source (without .env)
└── Akbarxon_AI_Living_Advisor_Uzbekistan_Updated_Colab.ipynb
```

## Run locally

```bash
python -m venv .venv
# Windows: .venv\Scripts\activate
# macOS/Linux: source .venv/bin/activate
pip install -r requirements.txt
streamlit run app.py
```

## Deploy on Streamlit Community Cloud

1. Extract this ZIP and upload all repository contents to GitHub.
2. In Streamlit Community Cloud, create a new app from that GitHub repository.
3. Choose `app.py` as the main file.
4. Deploy. Streamlit will install dependencies from `requirements.txt`.

No paid API key is required for the core prototype.

## Prototype / responsible-use note

The city scores in `cities_uzbekistan.csv` are illustrative prototype values, not official statistics. Before public or production use, replace them with licensed, dated, auditable sources and cite each source. Housing, employment, safety, healthcare, immigration, legal, tax, and financial decisions should be independently verified.

The city cards and detailed profiles use the 16 city-specific photos supplied with this revision. The remaining hero/detail artwork from the original UI package is retained only for general interface design. Confirm image ownership, attribution, and publication rights before a public production release.

## Live app

Streamlit domain: `https://uzbekistan-ai-map-living-advisor.streamlit.app/`

A QR code for the live app is included at `assets/app_qr_code.png`.

## Streamlit Cloud header-overlap fix

This repository includes a layout fix for Streamlit Community Cloud's fixed native toolbar. The main content container now reserves a safe top spacing before the custom Akbarxon AI Living Advisor brand/navigation area, while the sidebar keeps its compact spacing. This prevents the application title and navigation from being hidden underneath the Streamlit toolbar on wide desktop layouts.


In [ ]:
%%writefile .gitignore
__pycache__/
*.py[cod]
.venv/
.env
.streamlit/secrets.toml
feedback_state.json
demo_preference_dnn.joblib
.DS_Store


In [ ]:
from pathlib import Path

Path('.streamlit').mkdir(exist_ok=True)
Path('.streamlit/config.toml').write_text(
    '''[theme]
base = "light"
primaryColor = "#ff4b5c"
backgroundColor = "#ffffff"
secondaryBackgroundColor = "#f3f6fb"
textColor = "#172033"
font = "sans serif"

[server]
headless = true

[browser]
gatherUsageStats = false
''',
    encoding='utf-8',
)
print('Wrote .streamlit/config.toml')


## 5. Validate the generated project

This cell checks Python syntax, verifies the required files, trains the demonstration neural network, and runs two recommendation examples.

In [ ]:
import py_compile
from pathlib import Path

required_files = [
    Path('app.py'),
    Path('cities_uzbekistan.csv'),
    Path('requirements.txt'),
    Path('README.md'),
    Path('.gitignore'),
]

for file_path in required_files:
    assert file_path.exists(), f'Missing required file: {file_path}'

py_compile.compile('app.py', doraise=True)
print('✅ app.py syntax validation passed.')
print('✅ All required project files exist.')

In [ ]:
import app

city_df = app.load_city_data()
feedback = app.load_feedback_state(city_df['city'])
zero_sliders = {feature: 0.0 for feature in app.FEATURE_COLUMNS}

example_queries = [
    'cheapest living city with the cheapest grocery price',
    'I want a city with perfect sunset views and great entertainment activities',
]

for query in example_queries:
    ranked, weights, matches = app.rank_cities(
        city_df,
        query=query,
        slider_weights=zero_sliders,
        feedback_state=feedback,
        top_n=5,
    )
    print('
QUERY:', query)
    display(ranked[['city', 'region', 'match_score', 'preference_score', 'dnn_like_probability']])
    print('Detected intent:', matches)

print('✅ Recommendation smoke tests completed.')

## 6. Package everything as a GitHub-uploadable ZIP

This cell packages the current Streamlit application, the city dataset, all 16 city-specific photos, the original UI source, Streamlit configuration, and supporting documentation. Run it from the repository root so the referenced folders are present.


In [ ]:
import shutil
from pathlib import Path

repo_dir = Path("akbarxon-ai-living-advisor-city-photos-fixed")
if repo_dir.exists():
    shutil.rmtree(repo_dir)
repo_dir.mkdir(exist_ok=True)

for filename in [
    "app.py",
    "cities_uzbekistan.csv",
    "requirements.txt",
    "README.md",
    "UI_MIGRATION.md",
    "CITY_IMAGE_MAPPING.md",
    "CHANGELOG.md",
    ".gitignore",
    "Akbarxon_AI_Living_Advisor_Uzbekistan_Updated_Colab.ipynb",
]:
    source = Path(filename)
    if source.exists():
        shutil.copy2(source, repo_dir / filename)

for directory in [".streamlit", "assets", "ui_source"]:
    source = Path(directory)
    if source.exists():
        shutil.copytree(source, repo_dir / directory, dirs_exist_ok=True)

zip_path = shutil.make_archive(
    base_name="Akbarxon_AI_Living_Advisor_City_Photos_Fixed_GitHub_Repo",
    format="zip",
    root_dir=repo_dir,
)
print("✅ Created:", zip_path)


In [ ]:
# In Google Colab, run this cell to download the generated GitHub package.
try:
    from google.colab import files
    files.download("Akbarxon_AI_Living_Advisor_City_Photos_Fixed_GitHub_Repo.zip")
except ImportError:
    print("Not running in Colab. The ZIP file is available in the current working directory.")


## 7. Preview the Streamlit website inside Colab

Run the next cell. Colab will display a clickable proxy URL. Keep the cell's Streamlit process running while testing the app.

In [ ]:
import os
import subprocess
import time

# Stop an older preview process if this cell is rerun.
subprocess.run(['pkill', '-f', 'streamlit run app.py'], check=False)

log_file = open('streamlit.log', 'w')
process = subprocess.Popen(
    [
        'streamlit', 'run', 'app.py',
        '--server.port', '8501',
        '--server.headless', 'true',
        '--browser.gatherUsageStats', 'false',
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)
time.sleep(4)

try:
    from google.colab import output
    preview_url = output.eval_js('google.colab.kernel.proxyPort(8501)')
    print('Open the Streamlit preview:', preview_url)
except ImportError:
    print('Streamlit is running at http://localhost:8501')

print('Process ID:', process.pid)

In [ ]:
# Optional: inspect the Streamlit startup log if the preview does not open.
print(Path('streamlit.log').read_text(encoding='utf-8')[-4000:])

## 8. Upload to GitHub and deploy on Streamlit Community Cloud

1. Create a new public GitHub repository, for example `akbarxon-ai-living-advisor`.
2. Extract `akbarxon_ai_living_advisor_github.zip`.
3. Upload the extracted files to the repository root.
4. Open Streamlit Community Cloud and create a new app from that repository.
5. Select `app.py` as the entry-point file.
6. Deploy and test the map, recommendation examples, live context, and feedback controls.

### Files that must remain together

```text
akbarxon-ai-living-advisor/
├── app.py
├── cities_uzbekistan.csv
├── requirements.txt
├── README.md
└── .gitignore
```

### Recommended production upgrades

- Replace prototype city scores with licensed city-level housing, food, transport, healthcare, education, safety, air-quality, weather, and employment data.
- Store the source URL, source organization, retrieval date, geographic scope, and confidence for every metric.
- Replace local feedback JSON with PostgreSQL, Supabase, or Firebase.
- Add user accounts, consent, deletion requests, rate limiting, moderation, and audit logs.
- Add Uzbek and Russian translations.
- Add a retrieval pipeline with a search API, source quality ranking, deduplication, citation generation, and stale-data detection.
- Evaluate recommendation accuracy, subgroup fairness, privacy, and harmful relocation advice before public release.

### Ethical and legal guardrails

The platform should never present prototype scores as facts or guarantee that a location is safe, cheap, medically appropriate, or suitable for immigration. Users should receive source links, dates, uncertainty indicators, and reminders to verify housing, employment, visa, healthcare, and legal information independently.

## 9. Technical references

- Google Colab runtime FAQ: current runtime images and Python versions
- Streamlit release notes and deployment documentation
- Scikit-learn documentation for TF–IDF, cosine similarity, K-means, and `MLPClassifier`
- Folium documentation for interactive Leaflet/OpenStreetMap maps
- Open-Meteo API documentation for public weather context
- MediaWiki REST API documentation for city summaries

The application deliberately uses an explainable hybrid architecture. The weighted score remains dominant, so users can understand why a city was recommended rather than receiving an unexplained black-box answer.